# Task 5: Continuous-Time Silverbox Baseline (AccelerationMLP + RK4)

This notebook implements the baseline experiment for modeling the Silverbox benchmark system using a neural ordinary differential equation (AccelerationMLP integrated with explicit RK4).

In [1]:
import os
import time
import random
import numpy as np
import scipy.io
import torch
import torch.nn as nn
import pandas as pd
import matplotlib.pyplot as plt

# Ensure reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Libraries loaded and seed initialized to", SEED)

Libraries loaded and seed initialized to 42


## Step 1: Load and Prepare Data

Load $V_1$ as input $u$ and $V_2$ as output $y$. Slice zero-based excerpt `[42650:46746]` (4096 samples), zero-mean independently, set sampling time $dt = 1/f_s$, and split into train (0:3072) and test (3072:4096).

In [2]:
DATA_PATH = r"C:\Users\saroy\Codes\Uni_HW\SilverboxFiles\SNLS80mV.mat"

mat_data = scipy.io.loadmat(DATA_PATH)
u_raw = mat_data["V1"].squeeze().astype(np.float32)
y_raw = mat_data["V2"].squeeze().astype(np.float32)

assert np.all(np.isfinite(u_raw)), "Non-finite values found in raw input u"
assert np.all(np.isfinite(y_raw)), "Non-finite values found in raw output y"
assert len(u_raw) >= 46746 and len(y_raw) >= 46746, f"Insufficient samples: {len(u_raw)}"

# Extract slice [42650:46746]
u_slice = u_raw[42650:46746]
y_slice = y_raw[42650:46746]
assert len(u_slice) == 4096 and len(y_slice) == 4096, "Slice must be exactly 4096 samples"

# Independent zero-mean subtraction
u_full = u_slice - np.mean(u_slice)
y_full = y_slice - np.mean(y_slice)

fs = 610.35
dt = float(1.0 / fs)

u_train = u_full[:3072]
y_train = y_full[:3072]
u_test = u_full[3072:4096]
y_test = y_full[3072:4096]

print(f"Data prepared: u_full {u_full.shape}, y_full {y_full.shape}, dt = {dt:.6e} s")
print(f"Train samples: {len(u_train)}, Test samples: {len(u_test)}")

Data prepared: u_full (4096,), y_full (4096,), dt = 1.638404e-03 s
Train samples: 3072, Test samples: 1024


## Step 2: Training-Only Scales

Compute physical normalization scales strictly using the training excerpt and numerical finite differences.

In [3]:
y_scale = float(max(np.std(y_train), 1e-6))
u_scale = float(max(np.std(u_train), 1e-6))

v_train_est = np.diff(y_train) / dt
v_scale = float(max(np.std(v_train_est), 1e-6))

a_train_est = np.diff(v_train_est) / dt
a_scale = float(max(np.std(a_train_est), 1e-6))

print(f"Scales computed from train set:")
print(f"  y_scale = {y_scale:.6e}")
print(f"  v_scale = {v_scale:.6e}")
print(f"  u_scale = {u_scale:.6e}")
print(f"  a_scale = {a_scale:.6e}")

Scales computed from train set:
  y_scale = 5.301930e-02
  v_scale = 2.218151e+01
  u_scale = 2.195673e-02
  a_scale = 9.886329e+03


## Step 3: Model and Differentiable RK4 Integrator

Define `AccelerationMLP` (layers 3 -> 64 -> 64 -> 1 with Tanh) and explicit differentiable `rk4_step`.

In [4]:
class AccelerationMLP(nn.Module):
    def __init__(self, y_scale: float, v_scale: float, u_scale: float, a_scale: float):
        super().__init__()
        self.register_buffer("y_scale", torch.tensor(y_scale, dtype=torch.float32))
        self.register_buffer("v_scale", torch.tensor(v_scale, dtype=torch.float32))
        self.register_buffer("u_scale", torch.tensor(u_scale, dtype=torch.float32))
        self.register_buffer("a_scale", torch.tensor(a_scale, dtype=torch.float32))
        
        self.net = nn.Sequential(
            nn.Linear(3, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
    
    def forward(self, y: torch.Tensor, v: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
        y_norm = y / self.y_scale
        v_norm = v / self.v_scale
        u_norm = u / self.u_scale
        inp = torch.stack([y_norm, v_norm, u_norm], dim=-1)
        out = self.net(inp).squeeze(-1)
        return out * self.a_scale

def rk4_step(model: nn.Module, y: torch.Tensor, v: torch.Tensor, u_curr: torch.Tensor, u_next: torch.Tensor, dt: float):
    u_mid = 0.5 * (u_curr + u_next)
    
    k1_y = v
    k1_v = model(y, v, u_curr)
    
    y_mid1 = y + 0.5 * dt * k1_y
    v_mid1 = v + 0.5 * dt * k1_v
    k2_y = v_mid1
    k2_v = model(y_mid1, v_mid1, u_mid)
    
    y_mid2 = y + 0.5 * dt * k2_y
    v_mid2 = v + 0.5 * dt * k2_v
    k3_y = v_mid2
    k3_v = model(y_mid2, v_mid2, u_mid)
    
    y_end = y + dt * k3_y
    v_end = v + dt * k3_v
    k4_y = v_end
    k4_v = model(y_end, v_end, u_next)
    
    y_next = y + (dt / 6.0) * (k1_y + 2.0 * k2_y + 2.0 * k3_y + k4_y)
    v_next = v + (dt / 6.0) * (k1_v + 2.0 * k2_v + 2.0 * k3_v + k4_v)
    return y_next, v_next

model = AccelerationMLP(y_scale, v_scale, u_scale, a_scale)
print(model)

AccelerationMLP(
  (net): Sequential(
    (0): Linear(in_features=3, out_features=64, bias=True)
    (1): Tanh()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): Tanh()
    (4): Linear(in_features=64, out_features=1, bias=True)
  )
)


## Step 4: Baseline Training with Truncated Rollouts

Train for at most 800 updates or 175 seconds. Truncated rollout length is 32 samples with Adam ($lr=10^{-3}$) and gradient clipping at norm 1.0.

In [5]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

u_train_t = torch.tensor(u_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

rollout_len = 32
max_updates = 800
max_seconds = 175.0

train_start_time = time.time()
updates = 0

model.train()
while updates < max_updates:
    elapsed = time.time() - train_start_time
    if elapsed >= max_seconds:
        print(f"Time limit reached at update {updates}: {elapsed:.2f} s")
        break
    
    # Random segment wholly inside training set (0:3072)
    start_idx = random.randint(0, len(y_train) - rollout_len - 1)
    
    u_sec = u_train_t[start_idx : start_idx + rollout_len + 1]
    y_sec = y_train_t[start_idx : start_idx + rollout_len + 1]
    
    y_curr = y_sec[0]
    v_curr = (y_sec[1] - y_sec[0]) / dt
    
    y_preds = []
    for t in range(rollout_len):
        y_curr, v_curr = rk4_step(model, y_curr, v_curr, u_sec[t], u_sec[t+1], dt)
        y_preds.append(y_curr)
    
    y_preds_stack = torch.stack(y_preds)
    loss = loss_fn(y_preds_stack, y_sec[1:])
    
    if not torch.isfinite(loss):
        raise RuntimeError(f"Loss became non-finite ({loss.item()}) at update {updates}")
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    
    updates += 1
    if updates % 100 == 0 or updates == max_updates:
        print(f"Update {updates:3d}/{max_updates} | Loss: {loss.item():.6e} | Elapsed: {time.time() - train_start_time:.2f} s")

training_seconds = float(time.time() - train_start_time)
print(f"Training finished: {updates} updates in {training_seconds:.2f} seconds")

Update 100/800 | Loss: 7.318467e-03 | Elapsed: 5.85 s


Update 200/800 | Loss: 5.017232e-04 | Elapsed: 11.75 s


Update 300/800 | Loss: 4.804711e-05 | Elapsed: 17.99 s


Update 400/800 | Loss: 3.379505e-04 | Elapsed: 24.39 s


Update 500/800 | Loss: 1.541997e-04 | Elapsed: 30.84 s


Update 600/800 | Loss: 1.059510e-04 | Elapsed: 36.74 s


Update 700/800 | Loss: 1.327200e-04 | Elapsed: 42.69 s


Update 800/800 | Loss: 1.652396e-03 | Elapsed: 48.69 s
Training finished: 800 updates in 48.69 seconds


## Step 5: Continuous Rollout Evaluation

Execute a single continuous 4096-step rollout without feeding measured outputs or resetting at the split boundary.

In [6]:
model.eval()
u_full_t = torch.tensor(u_full, dtype=torch.float32)

with torch.no_grad():
    y_curr = torch.tensor(y_full[0], dtype=torch.float32)
    v_curr = torch.tensor((y_full[1] - y_full[0]) / dt, dtype=torch.float32)
    
    y_eval_preds = [float(y_curr.item())]
    for t in range(len(u_full) - 1):
        y_curr, v_curr = rk4_step(model, y_curr, v_curr, u_full_t[t], u_full_t[t+1], dt)
        y_eval_preds.append(float(y_curr.item()))

y_pred_full = np.array(y_eval_preds, dtype=np.float32)

assert len(y_pred_full) == 4096, f"Expected 4096 predictions, got {len(y_pred_full)}"
if not np.all(np.isfinite(y_pred_full)):
    raise RuntimeError("Uninterrupted rollout produced non-finite predictions.")

train_rmse = float(np.sqrt(np.mean((y_pred_full[:3072] - y_full[:3072]) ** 2)))
test_rmse = float(np.sqrt(np.mean((y_pred_full[3072:4096] - y_full[3072:4096]) ** 2)))
zero_test_rmse = float(np.sqrt(np.mean(y_test ** 2)))

print("Baseline Settings and Evaluation Metrics:")
print(f"  Architecture: 3 -> 64 -> 64 -> 1 (Tanh)")
print(f"  Learning Rate: 1e-3 | Rollout Length: 32")
print(f"  Updates Completed: {updates} | Training Time: {training_seconds:.2f} s")
print(f"  Train RMSE:      {train_rmse:.6f}")
print(f"  Test RMSE:       {test_rmse:.6f}")
print(f"  Zero Test RMSE:  {zero_test_rmse:.6f}")

Baseline Settings and Evaluation Metrics:
  Architecture: 3 -> 64 -> 64 -> 1 (Tanh)
  Learning Rate: 1e-3 | Rollout Length: 32
  Updates Completed: 800 | Training Time: 48.69 s
  Train RMSE:      0.034713
  Test RMSE:       0.039680
  Zero Test RMSE:  0.060492


## Step 6: Artifacts Generation and Marker Output

Save `comparison.csv` and `test_prediction.png`, and report the final verification markers.

In [7]:
# 1. Save comparison.csv
comparison_df = pd.DataFrame([{
    "model": "AccelerationMLP",
    "hidden_layers": "64-64",
    "learning_rate": 1e-3,
    "rollout_length": 32,
    "updates": updates,
    "training_seconds": round(training_seconds, 2),
    "train_rmse": round(train_rmse, 6),
    "test_rmse": round(test_rmse, 6),
    "zero_test_rmse": round(zero_test_rmse, 6),
    "status": "SUCCESS"
}])
comparison_df.to_csv("comparison.csv", index=False)
print("Saved comparison.csv:")
print(comparison_df.to_string())

# 2. Save test_prediction.png
plt.figure(figsize=(12, 5))
test_indices = np.arange(3072, 4096)
plt.plot(test_indices, y_full[3072:4096], label="Measured y", color="black", lw=1.2)
plt.plot(test_indices, y_pred_full[3072:4096], label="Predicted y", color="crimson", linestyle="--", lw=1.2)
plt.title("Silverbox Benchmark: Continuous Test Predictions (Samples 3072-4096)")
plt.xlabel("Sample Index")
plt.ylabel("Output Voltage (V)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig("test_prediction.png", dpi=300)
plt.close()
print("Saved test_prediction.png successfully.")

# 3. Final markers
print(f"TRAIN_RMSE={train_rmse:.6f}")
print(f"TEST_RMSE={test_rmse:.6f}")

Saved comparison.csv:
             model hidden_layers  learning_rate  rollout_length  updates  training_seconds  train_rmse  test_rmse  zero_test_rmse   status
0  AccelerationMLP         64-64          0.001              32      800             48.69    0.034713    0.03968        0.060492  SUCCESS


Saved test_prediction.png successfully.
TRAIN_RMSE=0.034713
TEST_RMSE=0.039680
